# Les 13 - Agentgeheugen


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Soorten Agentgeheugen

AI-agenten kunnen verschillende soorten geheugen gebruiken, elk met een specifiek doel:

### Werkgeheugen
De conversatiedraad zelf — de berichten die in een enkele sessie worden uitgewisseld. De agent kan terugverwijzen naar eerdere berichten in dezelfde draad om samenhang te behouden. In MAF wordt dit gemaakt via **`agent.create_session()`**, wat een `AgentSession` retourneert.

### Kortetermijngeheugen
Informatie die blijft bestaan gedurende de duur van een taak of sessie, maar niet permanent wordt opgeslagen. Bijvoorbeeld, de agent kan feiten verzamelen tijdens een planninggesprek met meerdere beurten en deze gebruiken om een definitieve reisroute op te stellen.

### Langetermijngeheugen
Voorkeuren en feiten die **sessieoverschrijdend** blijven bestaan. Een terugkerende gebruiker hoeft zijn dieetwensen of reisstijl niet te herhalen. Langetermijngeheugen wordt typisch ondersteund door een externe opslag — een database, bestand, of vectorindex — en aan de agent aangeboden via tools.


## Werkgeheugen met Sessies

De eenvoudigste vorm van geheugen is de conversatiesessie. Wanneer je dezelfde sessie (gemaakt via `agent.create_session()`) doorgeeft aan opeenvolgende `agent.run()`-aanroepen, ziet de agent de volledige geschiedenis van dat gesprek en kan hij eerdere details oproepen.

Laten we een reisagent maken en het werkgeheugen demonstreren.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

De agent herinnerde zich het budget correct omdat beide berichten dezelfde sessie delen. Dit is **werkgeheugen** — het bestaat alleen gedurende de levensduur van de sessie.

### Wat gebeurt er met een nieuwe thread?

Als we een **nieuwe** sessie maken, heeft de agent geen geheugen van het vorige gesprek:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Langdurig Geheugen Patroon

Om gebruikersvoorkeuren **over sessies heen** te onthouden, hebben we een persistente opslag nodig die buiten de gespreksdraad leeft. De agent krijgt toegang tot deze opslag via **tools** — functies die hij kan aanroepen om informatie op te slaan en op te halen.

Hieronder implementeren we een eenvoudige in-memory voorkeurenopslag (in productie zou je dit ondersteunen met een database of vectorindex) en stellen het bloot als tools die de agent kan gebruiken.

### Architectuur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Gebruiker die voor het eerst boekt een jubileumreis

Sarah bezoekt voor het eerst. De agent moet haar voorkeuren opslaan via de tools en deze gebruiken om hotels aan te raden.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah komt weken later terug

Sarah begint een **gloednieuw gesprek** (wat een nieuwe sessie simuleert). Het werkgeheugen is leeg, maar de langdurige voorkeurenopslag bevat nog steeds haar gegevens. De agent moet deze ophalen en gebruiken om aanbevelingen te personaliseren.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Samenvatting

In deze les heb je drie soorten agentgeheugen geleerd en hoe je ze kunt implementeren met het Microsoft Agent Framework:

| Geheugentype | MAF-mechanisme | Levensduur |
|---|---|---|
| **Werkgeheugen** | `agent.create_session()` | Enkele conversatie |
| **Kortetermijn** | Opgehoopte context binnen een thread | Enkele taak / sessie |
| **Langetermijn** | Externe opslag benaderd via `@tool` functies | Over sessies heen |

### Belangrijkste punten
1. **`agent.create_session()`** biedt werkgeheugen — de agent ziet de volledige gespreksgeschiedenis binnen een sessie.
2. **Nieuwe sessies verliezen context** — zonder langetermijngeheugen kan de agent zich geen eerdere gesprekken herinneren.
3. **`@tool` functies** overbruggen de kloof — ze laten de agent toe om informatie op te slaan en op te halen uit een persistente opslag.
4. **Personalisatie verbetert in de loop van de tijd** — hoe meer voorkeuren worden opgeslagen, hoe beter de aanbevelingen van de agent.

### Toepassingen in de praktijk
- **Klantenservice**: Herinner de geschiedenis en voorkeuren van klanten
- **Persoonlijke assistenten**: Behoud context over dagen of weken
- **Gezondheidszorg**: Volg patiëntinformatie en voorkeuren
- **E-commerce**: Gepersonaliseerd winkelen op basis van geschiedenis

### Volgende stappen
- Vervang het in-memory woordenboek door een database of vectorstore (bijv. Azure AI Search)
- Voeg geheugenverval toe voor tijdgevoelige informatie
- Bouw multi-agent systemen met gedeeld geheugen
- Verken het Cognee-notebook voor kennisgraaf-ondersteund geheugen


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
